In [1]:
import random
import time
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Activation, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd


In [9]:
from pathlib import Path
import gc
import json
import random
import time
import warnings
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
SOURCE_COLUMN = 'source_pcap'
LABEL_COLUMN = 'label'
BINARY_COLUMN = 'binary_label'
RANDOM_STATE = 42
OUTPUT_DIR = Path('unsw_nb15_prepared')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path('unsw_nb15_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEEDS = [11, 22, 33, 44, 55]
SOURCE_ROLE = {'unsw-nb15': 'same_domain_reextraction', 'darpa': 'heterogeneous', 'cic2017': 'heterogeneous', 'cic2018': 'heterogeneous'}


In [10]:
UNSW_FEATURE_COLUMNS = ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']


In [11]:
def source_label_distribution(frame, label_column=LABEL_COLUMN):
    table = frame.groupby([SOURCE_COLUMN, label_column], dropna=False).size().reset_index(name='count')
    table['within_source_pct'] = 100 * table['count'] / table.groupby(SOURCE_COLUMN)['count'].transform('sum')
    return table.sort_values([SOURCE_COLUMN, 'count'], ascending=[True, False])
TARGET_TEST_ATTACK_FRACTION = 0.551
TARGET_TEST_BENIGN_FRACTION = 1.0 - TARGET_TEST_ATTACK_FRACTION

def _largest_remainder_allocation(counts, requested_total):
    """Allocate a sample across strata while preserving proportions."""
    counts = pd.Series(counts, dtype='int64')
    available_total = int(counts.sum())
    if requested_total < 0 or requested_total > available_total:
        raise ValueError(f'requested_total={requested_total} exceeds available_total={available_total}')
    if requested_total == 0:
        return counts * 0
    raw = counts / available_total * requested_total
    allocation = np.floor(raw).astype('int64')
    remaining = int(requested_total - allocation.sum())
    fractional = (raw - allocation).sort_values(ascending=False)
    while remaining > 0:
        progressed = False
        for stratum in fractional.index:
            if allocation.loc[stratum] < counts.loc[stratum]:
                allocation.loc[stratum] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError('Could not complete proportional allocation.')
    return allocation

def proportional_stratified_sample(frame, requested_total, strata_column, random_state):
    if requested_total == len(frame):
        return frame.copy()
    strata = frame[strata_column].astype('string').fillna('<missing>')
    counts = strata.value_counts(sort=False)
    allocation = _largest_remainder_allocation(counts, requested_total)
    sampled_parts = []
    ordered_strata = sorted(allocation.index.astype(str))
    for offset, stratum in enumerate(ordered_strata):
        take = int(allocation.loc[stratum])
        if take == 0:
            continue
        cell = frame.loc[strata.eq(stratum)]
        sampled_parts.append(cell.sample(n=take, replace=False, random_state=random_state + offset))
    if not sampled_parts:
        return frame.iloc[0:0].copy()
    return pd.concat(sampled_parts, axis=0)

def match_binary_test_ratio(frame, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=RANDOM_STATE):
    if not 0.0 < target_attack_fraction < 1.0:
        raise ValueError('target_attack_fraction must be between 0 and 1.')
    working = frame.copy()
    working['__ratio_original_order'] = np.arange(len(working))
    attack_rows = working[working[BINARY_COLUMN].eq('attack')]
    benign_rows = working[working[BINARY_COLUMN].eq('benign')]
    available_attack = len(attack_rows)
    available_benign = len(benign_rows)
    if available_attack == 0 or available_benign == 0:
        raise ValueError('Every ratio-matched test source must contain both attack and benign rows.')
    target_benign_fraction = 1.0 - target_attack_fraction
    maximum_total = min(int(np.floor(available_attack / target_attack_fraction)), int(np.floor(available_benign / target_benign_fraction)))
    while maximum_total > 1:
        selected_attack = int(round(maximum_total * target_attack_fraction))
        selected_benign = maximum_total - selected_attack
        if 0 < selected_attack <= available_attack and 0 < selected_benign <= available_benign:
            break
        maximum_total -= 1
    else:
        raise ValueError('Not enough rows to construct a matched test set.')
    sampled_attack = proportional_stratified_sample(attack_rows, selected_attack, LABEL_COLUMN, random_state)
    sampled_benign = benign_rows.sample(n=selected_benign, replace=False, random_state=random_state + 10000)
    matched = pd.concat([sampled_attack, sampled_benign], axis=0).sort_values('__ratio_original_order').drop(columns='__ratio_original_order').reset_index(drop=True)
    actual_attack_fraction = matched[BINARY_COLUMN].eq('attack').mean()
    audit = {'available_rows': int(len(frame)), 'available_attack': int(available_attack), 'available_benign': int(available_benign), 'selected_rows': int(len(matched)), 'selected_attack': int(selected_attack), 'selected_benign': int(selected_benign), 'target_attack_pct': 100.0 * target_attack_fraction, 'actual_attack_pct': 100.0 * actual_attack_fraction, 'actual_benign_pct': 100.0 * (1.0 - actual_attack_fraction)}
    if round(audit['actual_attack_pct'], 1) != round(audit['target_attack_pct'], 1):
        raise AssertionError(f'The matched test set does not round to the requested attack prevalence: {audit}')
    return (matched, audit)

def ratio_match_test_sets(test_sets, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=RANDOM_STATE):
    matched_sets = {}
    audit_rows = []
    for offset, (test_source, test_frame) in enumerate(test_sets.items()):
        matched, audit = match_binary_test_ratio(test_frame, target_attack_fraction=target_attack_fraction, random_state=random_state + 1000 * offset)
        matched['ratio_match'] = '55.1% attack / 44.9% benign'
        matched_sets[test_source] = matched
        audit_rows.append({'test_source': test_source, **audit})
    audit_frame = pd.DataFrame(audit_rows)
    return (matched_sets, audit_frame)


In [ ]:
PARTITION_DIR = Path('/content/drive/MyDrive/Array_submission/Prepared_partitions/splits') / 'unsw_nb15'
exp4_train_balanced = pd.read_csv(PARTITION_DIR / 'balanced_train.csv', low_memory=False)
exp4_external_train_balanced = exp4_train_balanced
exp4_external_test_natural = pd.read_csv(PARTITION_DIR / 'test_all_sources.csv', low_memory=False)
required_columns = set(UNSW_FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN])
for frame_name, frame in [('balanced_train', exp4_train_balanced), ('test_all_sources', exp4_external_test_natural)]:
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise KeyError(f'{frame_name} is missing columns: {missing}')
exp4_natural_test_sets = {source: part.reset_index(drop=True) for source, part in exp4_external_test_natural.groupby(SOURCE_COLUMN, sort=True)}
exp4_test_sets, exp4_test_ratio_manifest = ratio_match_test_sets(exp4_natural_test_sets, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=RANDOM_STATE)
print('Loaded balanced training set:', exp4_train_balanced.shape)
print('Loaded test partition:', exp4_external_test_natural.shape)

Loaded balanced training set: (65120, 53)
Loaded test partition: (27912, 53)


In [ ]:
partition_manifest = pd.read_csv(PARTITION_DIR / 'partition_manifest.csv')
training_distribution = pd.read_csv(PARTITION_DIR / 'balanced_training_distribution.csv')
test_distribution = pd.read_csv(PARTITION_DIR / 'test_distribution.csv')
exp4_test_ratio_manifest.to_csv(RESULTS_DIR / 'exp4_test_ratio_manifest.csv', index=False)
training_distribution.to_csv(RESULTS_DIR / 'balanced_training_distribution.csv', index=False)
test_distribution.to_csv(RESULTS_DIR / 'test_distribution_before_ratio_matching.csv', index=False)
for test_source, test_frame in exp4_test_sets.items():
    test_frame.to_csv(RESULTS_DIR / f'exp4_test_{test_source}.csv', index=False)
pd.concat(exp4_test_sets.values(), ignore_index=True).to_csv(RESULTS_DIR / 'exp4_test_all_sources.csv', index=False)

In [14]:
MODEL_REGISTRY = {}
MODEL_KEYS = []

def register_binary_model(name, fit_function, predict_function):
    if name in MODEL_REGISTRY:
        raise KeyError(f'Model already registered: {name}')
    MODEL_REGISTRY[name] = {'fit': fit_function, 'predict': predict_function}
    MODEL_KEYS.append(name)
    print(f'Registered binary model: {name}')


In [15]:
def calculate_binary_metrics(y_true, y_pred, y_probability=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {'n_test': int(len(y_true)), 'attack_prevalence': float(np.mean(np.asarray(y_true) == 1)), 'benign_prevalence': float(np.mean(np.asarray(y_true) == 0)), 'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'mcc': matthews_corrcoef(y_true, y_pred), 'false_positive_rate': fp / (fp + tn) if fp + tn else np.nan, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    result['roc_auc'] = roc_auc_score(y_true, y_probability) if y_probability is not None and len(np.unique(y_true)) == 2 else np.nan
    return result

def category_recall_table(frame, y_true, y_pred):
    rows = []
    labels = frame[LABEL_COLUMN].astype(str).to_numpy()
    for category in sorted(pd.unique(labels)):
        mask = labels == category
        yt, yp = (y_true[mask], y_pred[mask])
        rows.append({'test_category': category, 'support': int(mask.sum()), 'attack_recall': recall_score(yt, yp, pos_label=1, zero_division=0) if np.any(yt == 1) else np.nan, 'benign_specificity': recall_score(yt, yp, pos_label=0, zero_division=0) if np.any(yt == 0) else np.nan})
    return pd.DataFrame(rows)


In [16]:
def evaluate_model_on_test_sets(model_key, train_frame, test_sets, experiment, training_variant, seed):
    adapter = MODEL_REGISTRY[model_key]
    print(f'{experiment} | {model_key} | {training_variant} | seed={seed} | TRAINING ONCE', flush=True)
    started = time.perf_counter()
    fitted = adapter['fit'](train_frame, seed)
    fit_seconds = time.perf_counter() - started
    overall_rows, category_rows, pooled_parts = ([], [], [])
    for test_source, test_frame in test_sets.items():
        print(f'  Testing {test_source}: {len(test_frame):,} rows', flush=True)
        y_true, y_pred, y_probability, evaluated = adapter['predict'](fitted, test_frame)
        metrics = calculate_binary_metrics(y_true, y_pred, y_probability)
        overall_rows.append({'experiment': experiment, 'training_variant': training_variant, 'model': model_key, 'seed': seed, 'test_source': test_source, 'source_role': SOURCE_ROLE.get(test_source, 'native_original'), 'fit_seconds': fit_seconds, **metrics})
        categories = category_recall_table(evaluated, y_true, y_pred)
        categories['experiment'] = experiment
        categories['training_variant'] = training_variant
        categories['model'] = model_key
        categories['seed'] = seed
        categories['test_source'] = test_source
        category_rows.append(categories)
        pooled_parts.append(pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'y_probability': y_probability, 'test_source': test_source}))
        print(f'    accuracy={metrics['accuracy']:.4f} f1={metrics['f1']:.4f} recall={metrics['recall']:.4f}', flush=True)
    pooled = pd.concat(pooled_parts, ignore_index=True)
    pooled_metrics = calculate_binary_metrics(pooled['y_true'].to_numpy(), pooled['y_pred'].to_numpy(), pooled['y_probability'].to_numpy())
    overall_rows.append({'experiment': experiment, 'training_variant': training_variant, 'model': model_key, 'seed': seed, 'test_source': 'POOLED_ALL_ROWS', 'source_role': 'pooled_row_weighted', 'fit_seconds': fit_seconds, **pooled_metrics})
    del fitted
    gc.collect()
    return (pd.DataFrame(overall_rows), pd.concat(category_rows, ignore_index=True))


In [17]:
def _merge_save(path, new_data, keys):
    if path.exists():
        old = pd.read_csv(path)
        combined = pd.concat([old, new_data], ignore_index=True)
    else:
        combined = new_data.copy()
    combined = combined.drop_duplicates(subset=keys, keep='last')
    combined.to_csv(path, index=False)
    return combined

def run_experiment(experiment, train_variants, test_sets):
    if not MODEL_KEYS:
        raise RuntimeError('No binary literature models are registered. Add their fit/predict adapters, then call register_binary_model(...).')
    overall_path = RESULTS_DIR / f'{experiment}_all_runs_partial.csv'
    category_path = RESULTS_DIR / f'{experiment}_category_runs_partial.csv'
    completed = pd.read_csv(overall_path) if overall_path.exists() else pd.DataFrame()
    all_overall = completed.copy()
    all_categories = pd.read_csv(category_path) if category_path.exists() else pd.DataFrame()
    for model_key in MODEL_KEYS:
        for variant, train_frame in train_variants.items():
            for seed in SEEDS:
                expected = set(test_sets) | {'POOLED_ALL_ROWS'}
                if not completed.empty:
                    mask = completed['model'].eq(model_key) & completed['training_variant'].eq(variant) & completed['seed'].eq(seed)
                    done = set(completed.loc[mask, 'test_source'].astype(str))
                else:
                    done = set()
                if expected.issubset(done):
                    print(f'SKIP completed: {model_key}, {variant}, seed={seed}')
                    continue
                overall, categories = evaluate_model_on_test_sets(model_key, train_frame, test_sets, experiment, variant, seed)
                all_overall = _merge_save(overall_path, overall, ['experiment', 'training_variant', 'model', 'seed', 'test_source'])
                all_categories = _merge_save(category_path, categories, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_category'])
                completed = all_overall
    return (all_overall, all_categories)

def summarize_results(results):
    metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
    per_source = results.groupby(['experiment', 'training_variant', 'model', 'test_source'], dropna=False)[metrics].agg(['mean', 'std', 'min', 'max']).reset_index()
    non_pooled = results[results['test_source'] != 'POOLED_ALL_ROWS']
    source_macro_seed = non_pooled.groupby(['experiment', 'training_variant', 'model', 'seed'])[metrics].mean().reset_index()
    source_macro = source_macro_seed.groupby(['experiment', 'training_variant', 'model'])[metrics].agg(['mean', 'std']).reset_index()
    return (per_source, source_macro)


In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from scipy.ndimage import gaussian_filter1d
UNSW_CATEGORICAL_COLUMNS = ['proto', 'service', 'state']
UNSW_NUMERICAL_COLUMNS = [column for column in UNSW_FEATURE_COLUMNS if column not in UNSW_CATEGORICAL_COLUMNS]
MODEL_KEYS = ['selected_feature_mlp', 'vinayakumar_dnn', 'rf_selected_1dcnn', 'nb_embedding_svm', 'seq2seq_convlstm']
EPOCHS = {'vinayakumar_dnn': 20, 'rf_selected_1dcnn': 50, 'seq2seq_convlstm': 30}
BATCH_SIZES = {'vinayakumar_dnn': 64, 'rf_selected_1dcnn': 64, 'seq2seq_convlstm': 32}
VALIDATION_SIZE = 0.2
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def binary_y(frame):
    if BINARY_COLUMN in frame.columns:
        return frame[BINARY_COLUMN].map({'benign': 0, 'attack': 1}).astype('int8').to_numpy()
    if 'official_binary_label' in frame.columns:
        return frame['official_binary_label'].astype('int8').to_numpy()
    return np.where(frame[LABEL_COLUMN].astype(str).str.casefold().eq('benign'), 0, 1).astype('int8')

def optional_sample(frame, maximum, seed):
    if maximum is None or len(frame) <= maximum:
        return frame.reset_index(drop=True)
    sampled, _ = train_test_split(frame, train_size=maximum, random_state=seed, stratify=binary_y(frame))
    return sampled.reset_index(drop=True)

def prepare_test(frame, seed):
    return optional_sample(frame, MAX_TEST_ROWS, seed)


In [19]:
def fit_ordinal_minmax_preprocessor(train_frame):
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    numerical_imputer = SimpleImputer(strategy='median')
    categorical_values = categorical_imputer.fit_transform(train_frame[UNSW_CATEGORICAL_COLUMNS])
    numerical_values = numerical_imputer.fit_transform(train_frame[UNSW_NUMERICAL_COLUMNS])
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
    categorical_encoded = encoder.fit_transform(categorical_values)
    combined = pd.DataFrame(index=train_frame.index)
    for index, column in enumerate(UNSW_CATEGORICAL_COLUMNS):
        combined[column] = categorical_encoded[:, index]
    for index, column in enumerate(UNSW_NUMERICAL_COLUMNS):
        combined[column] = numerical_values[:, index]
    combined = combined[UNSW_FEATURE_COLUMNS]
    scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
    X_train = scaler.fit_transform(combined).astype(np.float32)
    return ({'categorical_imputer': categorical_imputer, 'numerical_imputer': numerical_imputer, 'encoder': encoder, 'scaler': scaler}, X_train)

def transform_ordinal_minmax(preprocessor, frame):
    categorical_values = preprocessor['categorical_imputer'].transform(frame[UNSW_CATEGORICAL_COLUMNS])
    numerical_values = preprocessor['numerical_imputer'].transform(frame[UNSW_NUMERICAL_COLUMNS])
    categorical_encoded = preprocessor['encoder'].transform(categorical_values)
    combined = pd.DataFrame(index=frame.index)
    for index, column in enumerate(UNSW_CATEGORICAL_COLUMNS):
        combined[column] = categorical_encoded[:, index]
    for index, column in enumerate(UNSW_NUMERICAL_COLUMNS):
        combined[column] = numerical_values[:, index]
    combined = combined[UNSW_FEATURE_COLUMNS]
    return preprocessor['scaler'].transform(combined).astype(np.float32)


In [20]:
MLP_SELECTED_FEATURES = ['sttl', 'ct_srv_dst', 'sbytes', 'smeansz', 'proto', 'ct_state_ttl', 'sloss', 'synack', 'ct_dst_src_ltm', 'dmeansz', 'ct_srv_src', 'service', 'ct_dst_sport_ltm', 'dbytes', 'dloss', 'state', 'tcprtt', 'ct_src_dport_ltm']

def build_selected_feature_gpu_mlp(input_dim):
    """
    TensorFlow/Keras equivalent of the reproduced sklearn MLPClassifier:
    one hidden layer with 150 ReLU units, Adam, initial LR 0.02,
    L2 regularization 0.0001, and binary sigmoid output.
    """
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(input_dim,)), tf.keras.layers.Dense(150, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.0001)), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.02), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_selected_feature_mlp(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    train_part, validation_part = train_test_split(train_frame, test_size=VALIDATION_SIZE, random_state=seed, stratify=binary_y(train_frame))
    preprocessor, X_train_all = fit_ordinal_minmax_preprocessor(train_part)
    X_validation_all = transform_ordinal_minmax(preprocessor, validation_part)
    selected_indices = [UNSW_FEATURE_COLUMNS.index(feature) for feature in MLP_SELECTED_FEATURES]
    X_train = X_train_all[:, selected_indices]
    X_validation = X_validation_all[:, selected_indices]
    model = build_selected_feature_gpu_mlp(input_dim=len(selected_indices))
    callbacks = [tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-06, verbose=1), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)]
    print('  Training selected-feature GPU MLP. GPUs:', tf.config.list_physical_devices('GPU'), flush=True)
    model.fit(X_train, binary_y(train_part), validation_data=(X_validation, binary_y(validation_part)), epochs=20, batch_size=256, shuffle=True, callbacks=callbacks, verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'selected_indices': selected_indices, 'seed': seed}

def predict_selected_feature_mlp(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_ordinal_minmax(fitted['preprocessor'], evaluated)[:, fitted['selected_indices']]
    probability = fitted['model'].predict(X_test, batch_size=2048, verbose=1).reshape(-1)
    prediction = (probability >= 0.5).astype('int8')
    return (binary_y(evaluated), prediction, probability, evaluated)


In [21]:
def build_vinayakumar_unsw_model():
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(41,)), tf.keras.layers.Dense(1024, activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.01), tf.keras.layers.Dense(768, activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.01), tf.keras.layers.Dense(512, activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.01), tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.01), tf.keras.layers.Dense(128, activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.01), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_vinayakumar_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    train_part, validation_part = train_test_split(train_frame, test_size=VALIDATION_SIZE, random_state=seed, stratify=binary_y(train_frame))
    categorical_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_df = train_part[UNSW_FEATURE_COLUMNS].copy()
    X_validation_df = validation_part[UNSW_FEATURE_COLUMNS].copy()
    X_train_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = categorical_encoder.fit_transform(X_train_df[UNSW_CATEGORICAL_COLUMNS])
    X_validation_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = categorical_encoder.transform(X_validation_df[UNSW_CATEGORICAL_COLUMNS])
    normalizer = Normalizer(norm='l2')
    X_train = normalizer.fit_transform(X_train_df.astype(np.float32)).astype(np.float32)
    X_validation = normalizer.transform(X_validation_df.astype(np.float32)).astype(np.float32)
    model = build_vinayakumar_unsw_model()
    print('  Training Vinayakumar DNN...', flush=True)
    model.fit(X_train, binary_y(train_part), validation_data=(X_validation, binary_y(validation_part)), epochs=EPOCHS['vinayakumar_dnn'], batch_size=BATCH_SIZES['vinayakumar_dnn'], shuffle=True, verbose=2)
    return {'model': model, 'encoder': categorical_encoder, 'normalizer': normalizer, 'seed': seed}

def predict_vinayakumar_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test_df = evaluated[UNSW_FEATURE_COLUMNS].copy()
    X_test_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = fitted['encoder'].transform(X_test_df[UNSW_CATEGORICAL_COLUMNS])
    X_test = fitted['normalizer'].transform(X_test_df.astype(np.float32)).astype(np.float32)
    probability = fitted['model'].predict(X_test, batch_size=1024, verbose=1).reshape(-1)
    prediction = (probability >= 0.5).astype('int8')
    return (binary_y(evaluated), prediction, probability, evaluated)


In [22]:
NB_SVM_MAX_TEST_ROWS = 20000

class NaiveBayesFeatureEmbedding:

    def __init__(self, grid_size=2048):
        self.grid_size = grid_size

    def fit(self, X, y):
        self.normal_prior = np.mean(y == 0)
        self.attack_prior = np.mean(y == 1)
        self.log_prior_ratio = np.log(self.normal_prior / self.attack_prior)
        self.grid_edges = np.linspace(-0.1, 1.1, self.grid_size + 1)
        self.grid_centres = (self.grid_edges[:-1] + self.grid_edges[1:]) / 2
        grid_width = self.grid_edges[1] - self.grid_edges[0]
        self.densities = []
        for feature_index in range(X.shape[1]):
            feature_densities = []
            for target in [0, 1]:
                values = X[y == target, feature_index]
                std = np.std(values, ddof=1)
                iqr = np.percentile(values, 75) - np.percentile(values, 25)
                scales = [value for value in [std, iqr / 1.34] if np.isfinite(value) and value > 0]
                scale = min(scales) if scales else grid_width * 2
                bandwidth = max(0.9 * scale * len(values) ** (-0.2), grid_width * 1.5)
                histogram, _ = np.histogram(values, bins=self.grid_edges)
                density = gaussian_filter1d(histogram.astype(float), sigma=bandwidth / grid_width, mode='constant', cval=0.0)
                density = density / (max(density.sum(), 1e-12) * grid_width)
                feature_densities.append(np.maximum(density, 1e-12))
            self.densities.append(feature_densities)
        return self

    def transform(self, X):
        embedded = np.empty_like(X, dtype=np.float64)
        for feature_index, (normal_density, attack_density) in enumerate(self.densities):
            normal_values = np.interp(X[:, feature_index], self.grid_centres, normal_density, left=1e-12, right=1e-12)
            attack_values = np.interp(X[:, feature_index], self.grid_centres, attack_density, left=1e-12, right=1e-12)
            embedded[:, feature_index] = self.log_prior_ratio + np.log(normal_values) - np.log(attack_values)
        return embedded

def fit_nb_embedding_svm(train_frame, seed):
    set_all_seeds(seed)
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    s1, s2 = train_test_split(train_frame, test_size=0.5, random_state=seed, stratify=binary_y(train_frame))
    preprocessor, X_s1 = fit_ordinal_minmax_preprocessor(s1)
    X_s2 = transform_ordinal_minmax(preprocessor, s2)
    embedding = NaiveBayesFeatureEmbedding().fit(X_s1, binary_y(s1))
    embedded_s2 = embedding.transform(X_s2)
    embedded_scaler = StandardScaler()
    embedded_s2 = embedded_scaler.fit_transform(embedded_s2)
    model = SVC(C=1.0, kernel='rbf', gamma='auto', probability=False, cache_size=4096)
    print('  Training NB-embedded SVM...', flush=True)
    model.fit(embedded_s2, binary_y(s2))
    return {'model': model, 'preprocessor': preprocessor, 'embedding': embedding, 'embedded_scaler': embedded_scaler, 'seed': seed}

def predict_nb_embedding_svm(fitted, test_frame):
    evaluated = optional_sample(test_frame, NB_SVM_MAX_TEST_ROWS, fitted['seed'])
    print(f'  NB-SVM testing on {len(evaluated):,} sampled rows', flush=True)
    X_test = transform_ordinal_minmax(fitted['preprocessor'], evaluated)
    embedded = fitted['embedding'].transform(X_test)
    embedded = fitted['embedded_scaler'].transform(embedded)
    decision = fitted['model'].decision_function(embedded)
    prediction = (decision >= 0).astype('int8')
    return (binary_y(evaluated), prediction, decision, evaluated)


In [23]:
def build_rf_cnn():
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(15, 1)), tf.keras.layers.Conv1D(64, 6, padding='same', activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.MaxPooling1D(pool_size=3, strides=2, padding='same'), tf.keras.layers.Conv1D(64, 6, padding='same', activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.MaxPooling1D(pool_size=3, strides=2, padding='same'), tf.keras.layers.Conv1D(64, 6, padding='same', activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.MaxPooling1D(pool_size=3, strides=2, padding='same'), tf.keras.layers.Flatten(), tf.keras.layers.Dense(64, activation='relu'), tf.keras.layers.Dense(64, activation='relu'), tf.keras.layers.Dense(2, activation='softmax')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=tf.keras.losses.CategoricalCrossentropy(), metrics=['accuracy'])
    return model

def fit_rf_selected_1dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    train_part, validation_part = train_test_split(train_frame, test_size=VALIDATION_SIZE, random_state=seed, stratify=binary_y(train_frame))
    preprocessor, X_train_all = fit_ordinal_minmax_preprocessor(train_part)
    X_validation_all = transform_ordinal_minmax(preprocessor, validation_part)
    selector = RandomForestClassifier(n_estimators=100, criterion='gini', max_features='sqrt', bootstrap=True, random_state=seed, n_jobs=-1)
    print('  Fitting Random-Forest feature selector...', flush=True)
    selector.fit(X_train_all, binary_y(train_part))
    selected_indices = np.argsort(selector.feature_importances_)[-15:][::-1]
    X_train = X_train_all[:, selected_indices].reshape(-1, 15, 1)
    X_validation = X_validation_all[:, selected_indices].reshape(-1, 15, 1)
    model = build_rf_cnn()
    print('  Training RF-selected 1D CNN...', flush=True)
    model.fit(X_train, tf.keras.utils.to_categorical(binary_y(train_part), num_classes=2), validation_data=(X_validation, tf.keras.utils.to_categorical(binary_y(validation_part), num_classes=2)), epochs=EPOCHS['rf_selected_1dcnn'], batch_size=BATCH_SIZES['rf_selected_1dcnn'], shuffle=True, verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'selected_indices': selected_indices, 'seed': seed}

def predict_rf_selected_1dcnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_ordinal_minmax(fitted['preprocessor'], evaluated)[:, fitted['selected_indices']].reshape(-1, 15, 1)
    probability_matrix = fitted['model'].predict(X_test, batch_size=1024, verbose=1)
    prediction = np.argmax(probability_matrix, axis=1).astype('int8')
    return (binary_y(evaluated), prediction, probability_matrix[:, 1], evaluated)


In [24]:
SEQUENCE_LENGTH = 8

def build_seq2seq_convlstm(n_features):
    model_input = tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, n_features, 1))
    x = tf.keras.layers.ConvLSTM1D(128, 4, padding='same', return_sequences=True)(model_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), padding='same')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.ConvLSTM1D(256, 4, padding='same', return_sequences=True)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    output = tf.keras.layers.Dense(2, activation='softmax')(x)
    model = tf.keras.Model(model_input, output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def make_sequence_dataset(X, y, shuffle, seed, batch_size):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int32)
    print(f'Sequence matrix: {X.shape[0]:,} rows × {X.shape[1]} features', flush=True)
    targets = tf.keras.utils.to_categorical(y[SEQUENCE_LENGTH - 1:], num_classes=2)
    return tf.keras.utils.timeseries_dataset_from_array(data=X[..., np.newaxis], targets=targets, sequence_length=SEQUENCE_LENGTH, sequence_stride=1, shuffle=shuffle, seed=seed, batch_size=batch_size).prefetch(tf.data.AUTOTUNE)

def fit_seq2seq_convlstm(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed).reset_index(drop=True)
    split_index = int(len(train_frame) * (1 - VALIDATION_SIZE))
    train_part = train_frame.iloc[:split_index].copy()
    validation_part = train_frame.iloc[split_index:].copy()
    preprocessor, X_train = fit_ordinal_minmax_preprocessor(train_part)
    X_validation = transform_ordinal_minmax(preprocessor, validation_part)
    y_train = binary_y(train_part)
    y_validation = binary_y(validation_part)
    train_dataset = make_sequence_dataset(X_train, y_train, shuffle=True, seed=seed, batch_size=BATCH_SIZES['seq2seq_convlstm'])
    validation_dataset = make_sequence_dataset(X_validation, y_validation, shuffle=False, seed=seed, batch_size=BATCH_SIZES['seq2seq_convlstm'])
    model = build_seq2seq_convlstm(n_features=X_train.shape[1])
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, min_delta=0.0001, restore_best_weights=True, verbose=1), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-06, verbose=1)]
    print(f'Training Seq2Seq ConvLSTM with {X_train.shape[1]} features...', flush=True)
    model.fit(train_dataset, validation_data=validation_dataset, epochs=EPOCHS['seq2seq_convlstm'], callbacks=callbacks, verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed, 'n_features': X_train.shape[1]}

def predict_seq2seq_convlstm(fitted, test_frame):
    evaluated_full = prepare_test(test_frame, fitted['seed']).reset_index(drop=True)
    X_test = transform_ordinal_minmax(fitted['preprocessor'], evaluated_full)
    y_test = binary_y(evaluated_full)
    test_dataset = make_sequence_dataset(X_test, y_test, shuffle=False, seed=fitted['seed'], batch_size=BATCH_SIZES['seq2seq_convlstm'])
    probabilities = fitted['model'].predict(test_dataset, verbose=1)
    predictions = np.argmax(probabilities, axis=1).astype('int8')
    evaluated = evaluated_full.iloc[SEQUENCE_LENGTH - 1:].reset_index(drop=True)
    return (y_test[SEQUENCE_LENGTH - 1:], predictions, probabilities[:, 1], evaluated)


In [25]:
MODEL_REGISTRY = {}
ALL_MODEL_KEYS = []

def register_binary_model(name, fit_function, predict_function):
    MODEL_REGISTRY[name] = {'fit': fit_function, 'predict': predict_function}
    ALL_MODEL_KEYS.append(name)
register_binary_model('selected_feature_mlp', fit_selected_feature_mlp, predict_selected_feature_mlp)
register_binary_model('vinayakumar_dnn', fit_vinayakumar_dnn, predict_vinayakumar_dnn)
register_binary_model('nb_embedding_svm', fit_nb_embedding_svm, predict_nb_embedding_svm)
register_binary_model('rf_selected_1dcnn', fit_rf_selected_1dcnn, predict_rf_selected_1dcnn)
register_binary_model('seq2seq_convlstm', fit_seq2seq_convlstm, predict_seq2seq_convlstm)
unknown = sorted(set(MODEL_KEYS) - set(MODEL_REGISTRY))
if unknown:
    raise KeyError(f'Unknown selected models: {unknown}')
print('Registered models:', ALL_MODEL_KEYS)
print('Enabled models:', MODEL_KEYS)


Registered models: ['selected_feature_mlp', 'vinayakumar_dnn', 'nb_embedding_svm', 'rf_selected_1dcnn', 'seq2seq_convlstm']
Enabled models: ['selected_feature_mlp', 'vinayakumar_dnn', 'rf_selected_1dcnn', 'nb_embedding_svm', 'seq2seq_convlstm']


In [ ]:
EXP4_EXPERIMENT = 'mixed_source_unsw'
exp4_overall, exp4_categories = run_experiment(EXP4_EXPERIMENT, {'mixed_source_balanced_65120': exp4_train_balanced}, exp4_test_sets)
exp4_per_source, exp4_source_macro = summarize_results(exp4_overall)

mixed_source_unsw | selected_feature_mlp | mixed_source_balanced_65120 | seed=11 | TRAINING ONCE
  Training selected-feature GPU MLP. GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Epoch 1/20
204/204 - 7s - 32ms/step - accuracy: 0.9273 - loss: 0.2258 - val_accuracy: 0.9611 - val_loss: 0.1540 - learning_rate: 0.0200
Epoch 2/20
204/204 - 1s - 3ms/step - accuracy: 0.9649 - loss: 0.1365 - val_accuracy: 0.9676 - val_loss: 0.1234 - learning_rate: 0.0200
Epoch 3/20
204/204 - 1s - 3ms/step - accuracy: 0.9704 - loss: 0.1199 - val_accuracy: 0.9693 - val_loss: 0.1154 - learning_rate: 0.0200
Epoch 4/20
204/204 - 1s - 3ms/step - accuracy: 0.9729 - loss: 0.1130 - val_accuracy: 0.9711 - val_loss: 0.1122 - learning_rate: 0.0200
Epoch 5/20
204/204 - 1s - 3ms/step - accuracy: 0.9741 - loss: 0.1095 - val_accuracy: 0.9703 - val_loss: 0.1097 - learning_rate: 0.0200
Epoch 6/20
204/204 - 1s - 3ms/step - accuracy: 0.9755 - loss: 0.1062 - val_accuracy: 0.9711 - val_loss: 0.1075 - lear

In [27]:
exp4_per_source, exp4_source_macro = summarize_results(exp4_overall)
exp4_category_summary = exp4_categories.groupby(['experiment', 'training_variant', 'model', 'test_source', 'test_category'], dropna=False).agg(support=('support', 'first'), attack_recall_mean=('attack_recall', 'mean'), attack_recall_std=('attack_recall', 'std'), benign_specificity_mean=('benign_specificity', 'mean'), benign_specificity_std=('benign_specificity', 'std')).reset_index()
result_columns = ['model', 'seed', 'test_source', 'source_role', 'fit_seconds', 'n_test', 'attack_prevalence', 'benign_prevalence', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'tn', 'fp', 'fn', 'tp', 'roc_auc']
source_order = ['cic2017', 'cic2018', 'darpa', 'unsw-nb15', 'POOLED_ALL_ROWS']
table_results = exp4_overall.copy()
table_results['__model'] = pd.Categorical(table_results['model'], categories=MODEL_KEYS, ordered=True)
table_results['__seed'] = pd.Categorical(table_results['seed'], categories=SEEDS, ordered=True)
table_results['__source'] = pd.Categorical(table_results['test_source'], categories=source_order, ordered=True)
table_results = table_results.sort_values(['__model', '__seed', '__source']).loc[:, result_columns]
expected_rows = len(MODEL_KEYS) * len(SEEDS) * len(source_order)
if len(table_results) != expected_rows:
    raise AssertionError(f'Expected {expected_rows} result rows, got {len(table_results)}.')
table_results.to_csv(RESULTS_DIR / 'all_runs_UNSW.csv', index=False)
exp4_overall.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_all_runs_with_metadata.csv', index=False)
exp4_categories.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_category_runs.csv', index=False)
exp4_per_source.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_per_source_summary.csv', index=False)
exp4_source_macro.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_source_macro_summary.csv', index=False)
exp4_category_summary.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_category_summary.csv', index=False)
exp4_test_ratio_manifest.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_test_ratio_manifest.csv', index=False)